# The experiment showed zero conversions. The join was wrong.

This case recreates an experiment-tracking issue from a product analytics assessment. The assessment data are not included. All rows below are synthetic and generated in this notebook.

The assignment is stored on an exposure event. The payment event does not carry the assignment. We will compare a broken event-level aggregation with a user-level join.

## 1. Generate synthetic event data

The simulation creates one experiment with control and test groups. Payment probability is set to 8% in control and 10% in test to make the join problem visible. These values are simulation inputs, not findings from the assessment.

In [ ]:
import json
import random
from collections import Counter, defaultdict

random.seed(17)
n_users = 10_000
experiment = 'checkout_test'
events = []
assignments = {}

for user_id in range(n_users):
    group = 'control' if user_id % 2 == 0 else 'test'
    assignments[user_id] = group
    events.append({
        'user_id': user_id,
        'event_type': 'experiment_exposure',
        'event_params': json.dumps({'experiment_name': experiment, 'experiment_group': group})
    })
    payment_probability = 0.08 if group == 'control' else 0.10
    if random.random() < payment_probability:
        events.append({
            'user_id': user_id,
            'event_type': 'payment_done',
            'event_params': '{}'
        })

event_counts = Counter(row['event_type'] for row in events)
print(f"Users: {n_users:,}")
print('Event rows:', dict(event_counts))

## 2. Reproduce the broken comparison

The flawed approach groups payment events by experiment fields inside the payment event. Those fields are empty. The payment totals therefore cannot match the exposure groups.

In [ ]:
exposure_counts = Counter()
payment_counts_by_event_params = Counter()

for row in events:
    params = json.loads(row['event_params'])
    if row['event_type'] == 'experiment_exposure':
        key = (params.get('experiment_name'), params.get('experiment_group'))
        exposure_counts[key] += 1
    elif row['event_type'] == 'payment_done':
        key = (params.get('experiment_name'), params.get('experiment_group'))
        payment_counts_by_event_params[key] += 1

for key, count in sorted(exposure_counts.items()):
    # The payment event has no experiment fields, so this lookup returns zero.
    broken_payments = payment_counts_by_event_params[key]
    print(key, 'exposed users:', count, 'payments reported:', broken_payments)

The zero is a join failure. It does not mean that no exposed user paid. Keep the assignment and outcome events separate, then connect them with a stable user key.

## 3. Join assignment to outcome by user

Count each assigned user once. Count each payer once. Compute the rate within each group.

In [ ]:
assignment_by_user = {}
payer_ids = set()

for row in events:
    params = json.loads(row['event_params'])
    if row['event_type'] == 'experiment_exposure':
        assignment_by_user[row['user_id']] = params['experiment_group']
    elif row['event_type'] == 'payment_done':
        payer_ids.add(row['user_id'])

results = defaultdict(lambda: {'assigned_users': 0, 'payers': 0})
for user_id, group in assignment_by_user.items():
    results[group]['assigned_users'] += 1
    if user_id in payer_ids:
        results[group]['payers'] += 1

for group in ('control', 'test'):
    row = results[group]
    rate = row['payers'] / row['assigned_users']
    print(f"{group}: {row['payers']:,} / {row['assigned_users']:,} = {rate:.2%}")

The corrected rates describe this generated sample. They do not prove that the test variant caused a change. A real experiment also needs a valid randomization check, a defined exposure and payment window, one assignment per user per experiment, and a plan for multiple tests.

## 4. Checks before using experiment results

- Confirm that assignment is random and balanced.
- Confirm that each user has one group per experiment.
- Define whether the analysis starts at assignment or exposure.
- Use the same payment window for both groups.
- Deduplicate payment outcomes at the user level.
- Review all planned comparisons and guardrail metrics.
- Log assignment and exposure in a consistent event schema.

## Takeaway

When an experiment report shows zero conversions, inspect the event schema and join keys before interpreting the result as user behavior.